In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_H.xpt"
csv_file = DATA_DIR / "DIQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 DIQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040",
    "DIQ160",
    "DIQ170",
    "DIQ172",
    "DIQ175A",
    "DIQ175B",
    "DIQ175C",
    "DIQ175D",
    "DIQ175E",
    "DIQ175F",
    "DIQ175G",
    "DIQ175H",
    "DIQ175I",
    "DIQ175J",
    "DIQ175K",
    "DIQ175L",
    "DIQ175M",
    "DIQ175N",
    "DIQ175O",
    "DIQ175P",
    "DIQ175Q",
    "DIQ175R",
    "DIQ175S",
    "DIQ175T",
    "DIQ175U",
    "DIQ175V",
    "DIQ175W",
    "DIQ175X",
    "DIQ180",
    "DIQ050",
    "DID060",
    "DIQ060U",
    "DIQ070",
    "DIQ230",
    "DIQ240",
    "DID250",
    "DID260",
    "DIQ260U",
    "DIQ275",
    "DIQ280",
    "DIQ291",
    "DIQ300S",
    "DIQ300D",
    "DID310S",
    "DID310D",
    "DID320",
    "DID330",
    "DID341",
    "DID350",
    "DIQ350U",
    "DIQ360",
    "DIQ080"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DIQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9770:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 54:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,770 rows x 54 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS WHOLE-NUMBER FIELDS
# ============================================================

#
# IMPORTANT:
#
# DIQ280 = What was your last A1C level?
#
# This is a genuine decimal field.
#
# Examples can include:
#
#     5.4
#     6.7
#     8.2
#     10.5
#
# It also contains special questionnaire codes:
#
#     777 = Refused
#     999 = Don't know
#
# Because genuine decimal observations exist in the field,
# keep the ENTIRE DIQ280 column as floating-point.
#
# DO NOT:
#
#     round DIQ280
#     convert DIQ280 to Int64
#     strip ".0" from DIQ280
#     use float_format during CSV export
#

DECIMAL_COLUMNS = [
    "DIQ280"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly one genuine decimal field."
    )


if len(INTEGER_COLUMNS) != 53:

    raise ValueError(
        "Expected exactly 53 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value before checking
# the exact pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 54 DIQ_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached DIQ_H.xpt pattern:
#
# DID250 =  11
# DID260 = 138
# DID341 = 211
# DID350 = 128
#
# TOTAL  = 488
#

EXPECTED_TINY_COUNTS = {
    "DID250": 11,
    "DID260": 138,
    "DID341": 211,
    "DID350": 128
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DIQ_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 488:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact 488-value tiny-zero "
    "pattern across 4 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT do:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Only restore the four validated variables.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 488:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents the corrected source against
# which all later fidelity checks are performed.
#
# The ONLY permitted changes are:
#
#     validated TINY_VALUE -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST be performed before converting to Int64.
#
# np.round() below is used ONLY for testing whether
# numbers are whole-number.
#
# It does NOT change any values.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern changed.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: DIQ280 is the only genuine "
    "decimal variable."
)


# ============================================================
# 11. DIQ280 DECIMAL DETAILS
# ============================================================

s = df["DIQ280"]


diq280_nonmissing = int(
    s.notna()
    .sum()
)


diq280_missing = int(
    s.isna()
    .sum()
)


diq280_values = (
    s.dropna()
    .to_numpy(
        dtype=float
    )
)


diq280_fractional_mask = ~np.isclose(
    diq280_values,
    np.round(diq280_values),
    rtol=0,
    atol=1e-12
)


diq280_fractional = int(
    diq280_fractional_mask.sum()
)


diq280_whole_looking = int(
    (~diq280_fractional_mask).sum()
)


print("\n--- DIQ280 DECIMAL DETAILS ---")


print(
    "Nonmissing:",
    f"{diq280_nonmissing:,}"
)


print(
    "Missing:",
    f"{diq280_missing:,}"
)


print(
    "Fractional observations:",
    f"{diq280_fractional:,}"
)


print(
    "Whole-looking observations:",
    f"{diq280_whole_looking:,}"
)


if diq280_nonmissing != 520:

    raise ValueError(
        "Unexpected DIQ280 nonmissing count."
    )


if diq280_missing != 9250:

    raise ValueError(
        "Unexpected DIQ280 missing count."
    )


if diq280_fractional != 188:

    raise ValueError(
        "Unexpected number of fractional DIQ280 values."
    )


print(
    "PASS: DIQ280 decimal structure validated."
)


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 13. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert 53 fields to nullable Int64.
#
# DO NOT convert DIQ280.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 53 whole-number/code fields "
    "converted to nullable Int64."
)


print(
    "PASS: DIQ280 remains a genuine "
    "decimal field."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. HELPER - RANGE + SPECIAL CODE CHECK
# ============================================================

def check_range_special(
    column,
    low,
    high,
    range_count,
    missing_count,
    special_counts=None,
    zero_count=None
):

    if special_counts is None:
        special_counts = {}


    s = df[column]


    actual_range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    if actual_range_count != range_count:

        raise ValueError(
            f"{column} {low}-{high} count differs.\n"
            f"Expected: {range_count:,}\n"
            f"Actual:   {actual_range_count:,}"
        )


    if actual_missing != missing_count:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {missing_count:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    if zero_count is not None:

        actual_zero = int(
            s.eq(0)
            .sum()
        )


        if actual_zero != zero_count:

            raise ValueError(
                f"{column} zero count differs.\n"
                f"Expected: {zero_count:,}\n"
                f"Actual:   {actual_zero:,}"
            )


    for code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(code)
            .sum()
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )


    print(
        f"PASS: {column} range/special codes validated."
    )


# ============================================================
# 16. CDC CHECK - DIQ010
# ============================================================

#
# Doctor told you have diabetes.
#

check_frequency(
    "DIQ010",
    {
        1: 737,
        2: 8841,
        3: 185,
        7: 1,
        9: 5
    },
    1
)


# ============================================================
# 17. CDC CHECK - DID040
# ============================================================

#
# Age first told had diabetes.
#
# 666 = Less than 1 year.
#

check_range_special(
    "DID040",
    low=1,
    high=79,
    range_count=719,
    missing_count=9033,
    special_counts={
        80: 8,
        666: 2,
        777: 0,
        999: 8
    }
)


# ============================================================
# 18. CDC CHECK - PREDIABETES/RISK VARIABLES
# ============================================================

check_frequency(
    "DIQ160",
    {
        1: 278,
        2: 6001,
        9: 8
    },
    3483
)


check_frequency(
    "DIQ170",
    {
        1: 873,
        2: 5577,
        9: 19
    },
    3301
)


check_frequency(
    "DIQ172",
    {
        1: 1702,
        2: 4703,
        9: 64
    },
    3301
)


# ============================================================
# 19. CDC CHECK - DIQ180
# ============================================================

check_frequency(
    "DIQ180",
    {
        1: 2842,
        2: 3469,
        9: 158
    },
    3301
)


# ============================================================
# 20. CDC CHECK - INSULIN / DIABETIC PILLS
# ============================================================

check_frequency(
    "DIQ050",
    {
        1: 220,
        2: 9545,
        7: 1,
        9: 2
    },
    2
)


check_range_special(
    "DID060",
    low=1,
    high=46,
    range_count=217,
    missing_count=9550,
    special_counts={
        666: 2,
        777: 0,
        999: 1
    }
)


check_frequency(
    "DIQ060U",
    {
        1: 20,
        2: 197
    },
    9553
)


check_frequency(
    "DIQ070",
    {
        1: 547,
        2: 651,
        9: 2
    },
    8570
)


# ============================================================
# 21. INSULIN ROUTING CHECK
# ============================================================

insulin_yes = (
    df["DIQ050"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


did060_present = (
    df["DID060"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        insulin_yes
        !=
        did060_present
    )
) != 0:

    raise ValueError(
        "DID060 routing does not match DIQ050=1."
    )


did060_regular_value = (
    df["DID060"]
    .between(
        1,
        46
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq060u_present = (
    df["DIQ060U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        did060_regular_value
        !=
        diq060u_present
    )
) != 0:

    raise ValueError(
        "DIQ060U routing does not match "
        "regular DID060 month/year values."
    )


print(
    "PASS: Insulin duration routing validated."
)


# ============================================================
# 22. CDC CHECK - DIQ230 / DIQ240
# ============================================================

check_frequency(
    "DIQ230",
    {
        1: 279,
        2: 59,
        3: 69,
        4: 102,
        5: 224,
        9: 4
    },
    9033
)


check_frequency(
    "DIQ240",
    {
        1: 565,
        2: 172
    },
    9033
)


# ============================================================
# 23. CDC CHECK - DID250
# ============================================================

#
# Past year number of doctor visits.
#
# 0 = None.
#

check_range_special(
    "DID250",
    low=1,
    high=24,
    range_count=551,
    missing_count=9205,
    special_counts={
        7777: 0,
        9999: 3
    },
    zero_count=11
)


# ============================================================
# 24. DID250 ROUTING CHECK
# ============================================================

doctor_yes = (
    df["DIQ240"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


did250_present = (
    df["DID250"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        doctor_yes
        !=
        did250_present
    )
) != 0:

    raise ValueError(
        "DID250 routing does not match DIQ240=1."
    )


print(
    "PASS: DID250 routing validated."
)


# ============================================================
# 25. CDC CHECK - DID260 / DIQ260U
# ============================================================

#
# DID260:
#
# 0 = Never checking blood glucose/sugar.
#

check_range_special(
    "DID260",
    low=1,
    high=210,
    range_count=591,
    missing_count=9035,
    special_counts={
        777: 0,
        999: 6
    },
    zero_count=138
)


check_frequency(
    "DIQ260U",
    {
        1: 381,
        2: 144,
        3: 45,
        4: 21
    },
    9179
)


# ============================================================
# 26. DID260 UNIT ROUTING
# ============================================================

did260_regular = (
    df["DID260"]
    .between(
        1,
        210
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq260u_present = (
    df["DIQ260U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        did260_regular
        !=
        diq260u_present
    )
) != 0:

    raise ValueError(
        "DIQ260U routing does not match "
        "regular DID260 values."
    )


print(
    "PASS: DID260/DIQ260U routing validated."
)


# ============================================================
# 27. CDC CHECK - DIQ275
# ============================================================

check_frequency(
    "DIQ275",
    {
        1: 520,
        2: 148,
        7: 1,
        9: 68
    },
    9033
)


# ============================================================
# 28. CDC CHECK - DIQ280
# ============================================================

#
# Genuine decimal A1C variable.
#
# CDC:
#
# 4.1-14 = 330
# 777    = Refused, 2
# 999    = Don't know, 188
# Missing = 9,250
#

diq280_range_count = int(
    (
        df["DIQ280"]
        .between(
            4.1,
            14
        )
        &
        ~df["DIQ280"].isin(
            [
                777,
                999
            ]
        )
    )
    .sum()
)


diq280_777 = int(
    df["DIQ280"]
    .eq(777)
    .sum()
)


diq280_999 = int(
    df["DIQ280"]
    .eq(999)
    .sum()
)


diq280_missing = int(
    df["DIQ280"]
    .isna()
    .sum()
)


print("\n--- DIQ280 ---")


print(
    "A1C 4.1-14:",
    diq280_range_count
)


print(
    "777 Refused:",
    diq280_777
)


print(
    "999 Don't know:",
    diq280_999
)


print(
    "Missing:",
    diq280_missing
)


if not (
    diq280_range_count == 330
    and
    diq280_777 == 2
    and
    diq280_999 == 188
    and
    diq280_missing == 9250
):

    raise ValueError(
        "DIQ280 differs from CDC release."
    )


print(
    "PASS: DIQ280 A1C values validated "
    "without rounding."
)


# ============================================================
# 29. DIQ280 ROUTING CHECK
# ============================================================

a1c_checked_yes = (
    df["DIQ275"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq280_present = (
    df["DIQ280"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        a1c_checked_yes
        !=
        diq280_present
    )
) != 0:

    raise ValueError(
        "DIQ280 routing does not match DIQ275=1."
    )


print(
    "PASS: DIQ280 populated exactly for "
    "DIQ275=1 records."
)


# ============================================================
# 30. CDC CHECK - DIQ291
# ============================================================

check_frequency(
    "DIQ291",
    {
        1: 155,
        2: 149,
        3: 16,
        4: 4,
        5: 6,
        6: 107,
        99: 83
    },
    9250
)


diq291_present = (
    df["DIQ291"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        a1c_checked_yes
        !=
        diq291_present
    )
) != 0:

    raise ValueError(
        "DIQ291 routing does not match DIQ275=1."
    )


print(
    "PASS: DIQ291 routing validated."
)


# ============================================================
# 31. SELECTED BLOOD-PRESSURE / LDL CHECKS
# ============================================================

check_range_special(
    "DIQ300S",
    low=88,
    high=255,
    range_count=456,
    missing_count=9038,
    special_counts={
        7777: 3,
        9999: 273
    }
)


check_range_special(
    "DIQ300D",
    low=5,
    high=175,
    range_count=439,
    missing_count=9038,
    special_counts={
        7777: 3,
        9999: 290
    }
)


check_range_special(
    "DID310S",
    low=70,
    high=180,
    range_count=209,
    missing_count=9038,
    special_counts={
        6666: 390,
        7777: 1,
        9999: 132
    }
)


check_range_special(
    "DID310D",
    low=5,
    high=120,
    range_count=205,
    missing_count=9038,
    special_counts={
        6666: 390,
        7777: 1,
        9999: 136
    }
)


check_range_special(
    "DID320",
    low=5,
    high=350,
    range_count=96,
    missing_count=9038,
    special_counts={
        5555: 70,
        6666: 36,
        7777: 3,
        9999: 527
    }
)


check_range_special(
    "DID330",
    low=2,
    high=250,
    range_count=62,
    missing_count=9144,
    special_counts={
        6666: 288,
        7777: 1,
        9999: 275
    }
)


# ============================================================
# 32. CDC CHECK - DID341
# ============================================================

#
# Past year times doctor checked feet.
#
# 0 = None.
#

check_range_special(
    "DID341",
    low=1,
    high=104,
    range_count=511,
    missing_count=9040,
    special_counts={
        7777: 0,
        9999: 8
    },
    zero_count=211
)


# ============================================================
# 33. CDC CHECK - DID350 / DIQ350U
# ============================================================

#
# DID350:
#
# 0 = None.
#

check_range_special(
    "DID350",
    low=1,
    high=100,
    range_count=597,
    missing_count=9040,
    special_counts={
        7777: 0,
        9999: 5
    },
    zero_count=128
)


check_frequency(
    "DIQ350U",
    {
        1: 423,
        2: 119,
        3: 39,
        4: 16
    },
    9173
)


# ============================================================
# 34. DID350 UNIT ROUTING
# ============================================================

did350_regular = (
    df["DID350"]
    .between(
        1,
        100
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


diq350u_present = (
    df["DIQ350U"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        did350_regular
        !=
        diq350u_present
    )
) != 0:

    raise ValueError(
        "DIQ350U routing does not match "
        "regular DID350 values."
    )


print(
    "PASS: DID350/DIQ350U routing validated."
)


# ============================================================
# 35. CDC CHECK - DIQ360 / DIQ080
# ============================================================

check_frequency(
    "DIQ360",
    {
        1: 81,
        2: 362,
        3: 106,
        4: 115,
        5: 62,
        9: 6
    },
    9038
)


check_frequency(
    "DIQ080",
    {
        1: 156,
        2: 569,
        9: 7
    },
    9038
)


# ============================================================
# 36. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded file contained no ordinary zeros.
#
# Therefore the final zero pattern must exactly match:
#
# DID250 = 11
# DID260 = 138
# DID341 = 211
# DID350 = 128
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated DIQ_H pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 488:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the 488 validated restorations."
)


# ============================================================
# 37. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Nothing besides:
#
#     488 validated TINY_VALUE -> 0
#
# may have changed numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 54 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 38. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     manual formatting of DIQ280
#
# This preserves the genuine A1C decimal field naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 39. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 40. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 54 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 41. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 42. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 43. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 44. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT globally check all 54 columns for ".0".
#
# DIQ280 is a genuine decimal field.
#
# Only the 53 proven integer/code fields must not
# contain unnecessary ".0".
#

print("\n--- INTEGER FIELD .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 53 whole-number/code fields."
)


# ============================================================
# 45. LEGITIMATE DIQ280 .0 TOKENS
# ============================================================

#
# INFORMATIONAL ONLY.
#
# DIQ280 is a genuine decimal field.
#
# Values such as:
#
#     6.0
#     7.0
#     10.0
#     14.0
#     777.0
#     999.0
#
# may legitimately appear with ".0" when the field is
# naturally serialized as floating-point.
#
# DO NOT strip these globally.
#

diq280_dot_zero_count = int(
    (
        raw_csv_df["DIQ280"]
        .ne("")
        &
        raw_csv_df["DIQ280"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE DIQ280 .0 TOKENS ---"
)


print(
    "DIQ280 values ending in '.0':",
    f"{diq280_dot_zero_count:,}"
)


print(
    "PASS: DIQ280 was not globally reformatted."
)


# ============================================================
# 46. EXACT DIQ280 DECIMAL FIDELITY
# ============================================================

expected_diq280 = (
    expected_df["DIQ280"]
    .to_numpy(
        dtype=float
    )
)


exported_diq280 = (
    roundtrip_df["DIQ280"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_diq280,
    exported_diq280,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "DIQ280 decimal fidelity failed."
    )


print(
    "PASS: Every DIQ280 value preserved exactly."
)


# ============================================================
# 47. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly matches "
    "the validated 488 restorations."
)


# ============================================================
# 48. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 49. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 50. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "DIQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Diabetes"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "DID250 zero restorations:",
    EXPECTED_TINY_COUNTS["DID250"]
)


print(
    "DID260 zero restorations:",
    EXPECTED_TINY_COUNTS["DID260"]
)


print(
    "DID341 zero restorations:",
    EXPECTED_TINY_COUNTS["DID341"]
)


print(
    "DID350 zero restorations:",
    EXPECTED_TINY_COUNTS["DID350"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "DIQ280 nonmissing:",
    f"{diq280_nonmissing:,}"
)


print(
    "DIQ280 genuine fractional observations:",
    f"{diq280_fractional:,}"
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "DIQ280 decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Legitimate '.0' in DIQ280: "
    "ALLOWED"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 DIQ_H XPT -> CSV
Rows: 9,770
Columns: 54
PASS: Structure = 9,770 rows x 54 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['DIQ280']
Whole-number/code fields: 53

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 488

Tiny values by variable:
DID341    211
DID260    138
DID350    128
DID250     11

PASS: Exact 488-value tiny-zero pattern across 4 variables validated.

--- VALIDATED ZERO RESTORATION ---
DID250: 11 restored to 0
DID260: 138 restored to 0
DID341: 211 restored to 0
DID350: 128 restored to 0

PASS: 488 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
DIQ280: 188 fractional observations

PASS: DIQ280 is the only genuine decimal variable.

--- DIQ280 DECIMAL DETAILS ---
Nonmissing: 520
Missing: 9,250
Fractional observations: 188
Whole-looking observations: 332
PASS: DIQ280 decimal structure validated.

--- SEQN ---
Minimum: 73